In [2]:
class DatabaseError(Exception):
    """数据库层异常"""
    pass
class UserServiceError(Exception):
    """服务层异常"""
    pass
def get_user_from_db(user_id):
    try:
        raise ConnectionRefusedError("Connection refused: 127.0.0.1:5432")
    except ConnectionRefusedError as e:
        raise DatabaseError(f"无法连接数据库查询用户{user_id}") from e
def get_user_service(user_id):
    try:
        get_user_from_db(user_id)
    except DatabaseError as e:
        raise UserServiceError(f"用户服务不可用") from None
def retry_wrapper(func,*args):
    try:
        return func(*args)
    except Exception:
        print("  重试中...")  
        return func(*args)
try:
    get_user_service("U-001")
except UserServiceError as e:
    print(f"捕获: {e}")
    print(f"__cause__: {e.__cause__}")
    print(f"__context__: {e.__context__}")
print()
try:
    get_user_from_db("U-001")
except DatabaseError as e:
    print(f"捕获: {e}")
    print(f"__cause__: {e.__cause__}")

捕获: 用户服务不可用
__cause__: None
__context__: 无法连接数据库查询用户U-001

捕获: 无法连接数据库查询用户U-001
__cause__: Connection refused: 127.0.0.1:5432


In [4]:
import sys
if sys.version_info < (3,11):
    print("⚠️ ExceptionGroup 需要 Python 3.11+，以下代码仅供阅读理解")
else:
    errors = []
    try:
        raise ValueError("任务1: 参数无效")
    except ValueError as e:
        errors.append(e)
    try:
        raise TimeoutError("任务2: 超时")
    except TimeoutError as e:
        errors.append(e)
if errors:
    eg = ExceptionGroup("并行任务部分失败", errors)   
    try:
        raise eg
    except* ValueError as eg_val:
        print(f"  处理 ValueError: {eg_val.exceptions}")
    except* TimeoutError as eg_timeout:
        print(f"  处理 TimeoutError: {eg_timeout.exceptions}")

  处理 ValueError: (ValueError('任务1: 参数无效'),)
  处理 TimeoutError: (TimeoutError('任务2: 超时'),)


In [ ]:
class Tracked:
    _instance_count = 0
    def __new__(cls, *args, **kwargs):
        print(f"  __new__ 被调用: cls={cls.__name__}")
        instance = super().__new__(cls)
        return instance
    def __init__(self,value):
        print(f"  __init__ 被调用: self.value={value}")
        self.value = value
print("=== 创建第一个实例 ===")
a = Tracked(10)
print(f"a.value = {a.value}")

print("\n=== 创建第二个实例 ===")
b = Tracked(20)
print(f"b.value = {b.value}")

print(f"\na is b: {a is b}")
print(f"type(a): {type(a)}")


=== 创建第一个实例 ===
  __new__ 被调用: cls=Tracked
  __init__ 被调用: self.value=10
a.value = 10

=== 创建第二个实例 ===
  __new__ 被调用: cls=Tracked
  __init__ 被调用: self.value=20
b.value = 20

a is b: False
type(a): <class '__main__.Tracked'>


In [11]:
class DatabaseConnection:
    _instance = None
    def __new__(cls,*args,**kwargs):
        if cls._instance is None:
            print("  [new] 创建唯一实例...")
            cls._instance = super().__new__(cls)
        else:
            print("  [new] 实例已存在，复用")
        return cls._instance
    def __init__(self,host='loaclhost',port=5432):
        if not hasattr(self,'_initialized'):
            print(f"  [init] 初始化连接: {host}:{port}")
            self.host = host
            self.port = port
            self._initialized = True
    def query(self,sql):
        print(f"  [query] 在 {self.host}:{self.port} 执行: {sql}")  
conn1 = DatabaseConnection("db1.example.com", 5432)
conn2 = DatabaseConnection("db2.example.com", 3306)
print(f"\nconn1 is conn2: {conn1 is conn2}")
conn1.query("SELECT 1")
conn2.query("SELECT 2")

  [new] 创建唯一实例...
  [init] 初始化连接: db1.example.com:5432
  [new] 实例已存在，复用

conn1 is conn2: True
  [query] 在 db1.example.com:5432 执行: SELECT 1
  [query] 在 db1.example.com:5432 执行: SELECT 2


In [14]:
class Typed:
    def __init__(self,name,expected_type):
        self.name = name
        self.expected_type = expected_type
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.name]
    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"{self.name} 必须是 {self.expected_type.__name__},"
                f"得到 {type(value).__name__}"
            )
        instance.__dict__[self.name] = value
    def __delete__(self, instance):
        del instance.__dict__[self.name]
class Product:
    name = Typed("name",str)
    price = Typed("price",float)
    quantity = Typed("quantity",int)
    def __init__(self,name,price,quantity):
        self.name = name
        self.price = price
        self.quantity = quantity
p = Product('MacBook',12999.0,10)
print(f'{p.name},{p.price},{p.quantity}')
try:
    p.price = '免费'
except TypeError as e:
    print(f"类型错误: {e}")
p.__dict__['name'] = '直接写入'
print(f'绕过描述符直接访问: {p.name}')

MacBook,12999.0,10
类型错误: price 必须是 float,得到 str
绕过描述符直接访问: 直接写入


In [17]:
class Logged:
    def __init__(self,name):
        self.name = name
    def __get__(self, instance, owner):
        if instance is None:
            return self
        value = instance.__dict__[self.name]
        print(f"  [LOG] 读取 {self.name} = {value!r}")
        return value
    def __set__(self,instance,value):
        old = instance.__dict__.get(self.name,'未设置')
        instance.__dict__[self.name] = value
        print(f"  [LOG] 设置 {self.name}: {old!r} -> {value!r}")
class User:
    username = Logged('username')
    email = Logged('email')
    def __init__(self,username,email):
        self.username = username
        self.email = email
print("=== 创建用户 ===")
u = User("alice", "alice@example.com")
print("\n=== 读取属性 ===")
print(f"用户名: {u.username}")
print(f"邮箱: {u.email}")
print("\n=== 修改属性 ===")
u.username = "bob"
u.email = "bob@example.com"


=== 创建用户 ===
  [LOG] 设置 username: '未设置' -> 'alice'
  [LOG] 设置 email: '未设置' -> 'alice@example.com'

=== 读取属性 ===
  [LOG] 读取 username = 'alice'
用户名: alice
  [LOG] 读取 email = 'alice@example.com'
邮箱: alice@example.com

=== 修改属性 ===
  [LOG] 设置 username: 'alice' -> 'bob'
  [LOG] 设置 email: 'alice@example.com' -> 'bob@example.com'


In [31]:
class PluginRegistry(type):
    def __init__(cls,name,bases,namespace):
        super().__init__(name,bases,namespace)
        if not hasattr(cls,'_plugins'):
            cls._plugins = {}
        else:
            cls._plugins[name.lower()] = cls
            print(f"  [registry] 注册插件: {name}")
class Plugin(metaclass=PluginRegistry):
    _plugins = {}
    @classmethod
    def get_plugin(cls,name):
        return cls._plugins.get(name.lower())
class JSONParser(Plugin):
    def parse(self,data):
        return f'JSON解析{data}'
class XMLParser(Plugin):
    def parse(self,data):
        return f'XML解析{data}'
class CSVParser(Plugin):
    def parse(self,data):
        return f'CSV解析{data}'
print(f"\n已注册插件: {list(Plugin._plugins.keys())}")  
parser = Plugin.get_plugin("jsonparser")()
print(parser.parse('{"key": "value"}'))

  [registry] 注册插件: Plugin
  [registry] 注册插件: JSONParser
  [registry] 注册插件: XMLParser
  [registry] 注册插件: CSVParser

已注册插件: ['plugin', 'jsonparser', 'xmlparser', 'csvparser']
JSON解析{"key": "value"}
